# Lesson 1: Simple ReAct Agent from Scratch

### 本节课的核心思路：手写一个最简单的 ReAct Agent

ReAct（Reasoning + Acting）是一种让 LLM 交替进行「思考（Thought）」「行动（Action）」「观察（Observation）」的循环模式：

1. 模型先输出 `Thought:`（思考要做什么）
2. 再输出 `Action: 工具名: 工具输入`，然后停下来（`PAUSE`）
3. 我们（外部代码）真正执行这个工具，把结果作为 `Observation:` 喂回给模型
4. 模型继续思考，直到给出最终 `Answer:`

这一课不用 LangGraph，而是用最原始的方式（一个 `while` 循环 + 正则表达式解析模型输出）来实现这个循环，目的是让你理解 LangGraph 之后帮你自动化的到底是什么。

In [ ]:
import openai
import re          # 用正则表达式解析模型输出中的 "Action: xxx: yyy" 这一行
import httpx
import os
from dotenv import load_dotenv
_=load_dotenv()    # 从 .env 读取 OPENAI_API_KEY 等环境变量，OpenAI() 客户端会自动读取该环境变量
from openai import OpenAI

In [ ]:
client=OpenAI()  # 创建 OpenAI 客户端，用于后面直接调用 chat.completions 接口（原生 SDK，不经过 LangChain）

In [ ]:
chat_completion=client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello world"}]  # 最基础的单轮对话调用，先验证 API 能跑通
)

In [ ]:
chat_completion.choices[0].message.content  # 从返回对象里取出第一个候选回复的文本内容

In [ ]:
class Agent:
    # 用一个类封装「带记忆的对话」：内部维护 self.messages 列表，每次调用都会把新一轮的
    # user/assistant 消息追加进去，这样模型每次调用都能看到完整的历史对话（多轮 ReAct 循环需要这个记忆）
    def __init__(self,system=""):
        self.system=system
        self.messages=[]
        if self.system:
            self.messages.append({"role": "system", "content": system})  # 把 ReAct 的系统提示词放进 system message
    def __call__(self,message):
        self.messages.append({"role": "user", "content": message})
        result=self.execute()
        self.messages.append({"role": "assistant", "content": result})
        # 【bug 修复】原来这里没有 return result，__call__ 执行完只是把结果存进了 self.messages，
        # 但函数本身没有返回值，Python 函数不写 return 默认返回 None。
        # 后面所有 `result = abot(...)` 拿到的都会是 None，导致 print(result)、
        # "Observation: {}".format(result) 全部打印/格式化出 "None"。
        # 修复方式：在末尾加上 return result，让调用方能拿到这一轮模型的回复文本。
        return result
    def execute(self):
        completion=client.chat.completions.create(
            model="gpt-4o",
            temperature=0,      # 温度设为 0，让 ReAct 的 Action 格式尽量稳定、可复现，减少随机性导致的解析失败
            messages=self.messages,
        )
        return completion.choices[0].message.content

下面这段 `prompt` 就是 ReAct 的核心：它用 few-shot 的方式教模型「必须按 Thought/Action/PAUSE/Observation/Answer 这个固定格式输出」，
这样外部代码才能用正则表达式稳定地解析出模型想调用哪个工具、传了什么参数。

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)  # 直接用 Python eval 执行算术表达式字符串，简单粗暴（生产环境不安全，仅作教学演示）

def average_dog_weight(name):
    # 注意这里判断逻辑是 `name in "固定品种字符串"`，即「判断 name 是不是这个品种全名的子串」，
    # 这样即使模型只传了品种的一部分（比如 "Collie"）也能匹配上 "Border Collie"。
    # 顺序很重要：更长/更具体的品种名要放在前面，否则短的子串可能被误判为其他品种。
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

know_actions={  # 工具名 -> 工具函数 的映射表，ReAct 循环靠这个表把模型输出的 Action 名字路由到真正的 Python 函数
    "calculate":calculate,
    "average_dog_weight":average_dog_weight
}

In [ ]:
abot=Agent(prompt)  # 创建一个新的 Agent 实例，system prompt 就是上面定义的 ReAct 提示词

In [ ]:
result = abot("How much does a toy poodle weigh?")  # 手动模拟 ReAct 第一步：模型应该会输出 Thought + Action，然后 PAUSE
print(result)

In [ ]:
result=average_dog_weight("Toy Poodle")  # 手动执行模型请求的 Action（真实环境里这一步应该由外部代码自动完成）
print(result)

In [ ]:
next_prompt = "Observation: {}".format(result)  # 把工具执行结果包装成 "Observation: ..." 格式，喂回给模型继续下一轮

In [ ]:
abot(next_prompt)  # 把 Observation 发给模型，模型看到后应该会直接输出最终 Answer

In [ ]:
abot.messages  # 查看完整的对话历史，可以直观看到 system/user/assistant 消息是如何一轮轮累积的

In [ ]:
abot = Agent(prompt)  # 重新创建一个全新的 Agent（清空历史消息），演示一个更复杂、需要两次工具调用+一次计算的问题

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)  # 第一轮：模型应该先查第一只狗（比如 border collie）的体重，输出 Action 后 PAUSE

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))  # 手动执行第一次工具调用，构造 Observation
print(next_prompt)

In [ ]:
abot(next_prompt)  # 第二轮：模型拿到第一只狗的体重后，应该接着去查第二只狗（scottish terrier）

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))  # 手动执行第二次工具调用
print(next_prompt)

In [ ]:
abot(next_prompt)  # 第三轮：模型已经拿到两只狗各自的体重，应该会输出 Action: calculate: 37 + 20 这样的算术请求

In [ ]:
next_prompt = "Observation: {}".format(eval("37 + 20"))  # 手动模拟执行 calculate 工具（正常应调用 calculate 函数，这里直接 eval 效果一样）
print(next_prompt)

In [ ]:
abot(next_prompt)  # 第四轮：模型拿到计算结果后，应该输出最终 Answer，至此手动模拟完整个 ReAct 循环

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action
# 两个捕获组：第 1 组匹配 Action 名（如 average_dog_weight），第 2 组匹配传给该 Action 的输入（如 Toy Poodle）
# 用 actions[0].groups() 才能拿到 (action, action_input) 这个二元组；.group() 不传参数返回的是整行匹配文本

In [ ]:
def query(question,max_turns=5):
    # 这个函数就是把上面手动做的事情自动化：循环调用模型 -> 解析 Action -> 执行工具 -> 把 Observation 喂回去，
    # 直到模型不再输出 Action（说明它给出了最终 Answer）或达到最大轮数为止
    i=0
    bot=Agent(prompt)
    next_prompt=question
    while i<max_turns:
        i+=1
        result=bot(next_prompt)
        print(result)
        actions=[
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # 【bug 修复】原来写的是 actions[0].group()（不带参数）。
            # re.Match.group() 不传参数时返回的是整个匹配到的字符串，
            # 例如 "Action: average_dog_weight: Toy Poodle" 这一整行文本，
            # 把一个字符串拆包给 action, action_input 两个变量会导致解包错误或结果不对。
            # 应该用 actions[0].groups()（复数），它返回的是所有捕获组组成的元组，
            # 即 ("average_dog_weight", "Toy Poodle")，才能正确拆包出 action 和 action_input。
            action,action_input=actions[0].groups()
            if action not in know_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = know_actions[action](action_input)  # 通过 know_actions 字典把 Action 名路由到真正的 Python 函数并执行
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return  # 模型没有再输出 Action，说明它已经给出最终 Answer，循环结束

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)  # 一行代码跑完整个 ReAct 循环，效果等价于前面手动执行的一连串 abot(...) 调用